In [1]:
  !pip install -q kaggle
# Upload your kaggle.json file first
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d balraj98/synthetic-objective-testing-set-sots-reside

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/balraj98/synthetic-objective-testing-set-sots-reside
License(s): other
100% 415M/415M [00:01<00:00, 281MB/s]



In [2]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ STILL NO GPU - Try switching the accelerator in settings again.")

CUDA Available: True
Device Name: Tesla T4


In [3]:
import torch
import torch.nn as nn

class AttentionBlock(nn.Module):
    def __init__(self, in_channels):
        super(AttentionBlock, self).__init__()
        # Channel Attention: Squeeze and Excitation style
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // 8, 1),
            nn.ReLU(),
            nn.Conv2d(in_channels // 8, in_channels, 1),
            nn.Sigmoid()
        )
        # Spatial Attention: Focus on 'where'
        self.sa = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.ca(x) * self.sa(x)

In [4]:
import zipfile
import os

# Unzip the dataset
local_zip = '/content/synthetic-objective-testing-set-sots-reside.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/content/sots_data')
zip_ref.close()

# Check the folders
print(os.listdir('/content/sots_data/'))

['outdoor', 'indoor', 'metadata_indoor.csv']


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

class SOTSDataset(Dataset):
    def __init__(self, hazy_dir, clear_dir, transform=None):
        self.hazy_dir = hazy_dir
        self.clear_dir = clear_dir
        self.hazy_images = sorted(os.listdir(hazy_dir))
        self.clear_images = sorted(os.listdir(clear_dir))
        self.transform = transform

    def __len__(self):
        return len(self.hazy_images)

    def __getitem__(self, idx):
        hazy_img_path = os.path.join(self.hazy_dir, self.hazy_images[idx])
        clear_img_path = os.path.join(self.clear_dir, self.clear_images[idx])

        hazy_image = Image.open(hazy_img_path).convert('RGB')
        clear_image = Image.open(clear_img_path).convert('RGB')

        if self.transform:
            hazy_image = self.transform(hazy_image)
            clear_image = self.transform(clear_image)

        return hazy_image, clear_image

# Basic transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiLevelAttention(nn.Module):
    def __init__(self, in_channels):
        super(MultiLevelAttention, self).__init__()
        # Channel Attention (CA)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.ca_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, 1, bias=False),
            nn.Sigmoid()
        )
        # Spatial Attention (SA)
        self.sa_conv = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Apply Channel Attention
        y_ca = x * self.ca_conv(self.avg_pool(x))
        # Apply Spatial Attention
        y_sa = y_ca * self.sa_conv(y_ca)
        return y_sa

In [8]:
class OVALNet(nn.Module):
    def __init__(self):
        super(OVALNet, self).__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        # Encoder
        self.enc1 = conv_block(3, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = conv_block(256, 512)

        # Attention Fusion Blocks (The "OVAL" part)
        self.att3 = MultiLevelAttention(256)
        self.att2 = MultiLevelAttention(128)
        self.att1 = MultiLevelAttention(64)

        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        # Final Output Layer
        self.final = nn.Conv2d(64, 3, kernel_size=1)

    def forward(self, x):
        # Encoder side
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bridge
        b = self.bottleneck(self.pool(e3))

        # Decoder with Attention-weighted skip connections
        d3 = self.up3(b)
        d3 = torch.cat([d3, self.att3(e3)], dim=1) # Attention here
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, self.att2(e2)], dim=1) # Attention here
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, self.att1(e1)], dim=1) # Attention here
        d1 = self.dec1(d1)

        return self.final(d1)

# Instantiate the model
model = OVALNet().to('cuda' if torch.cuda.is_available() else 'cpu')
print("OVAL-Net Architecture loaded successfully!")

OVAL-Net Architecture loaded successfully!


In [9]:
import os
print("Folders inside indoor:", os.listdir('/content/sots_data/indoor'))

Folders inside indoor: ['clear', 'hazy']


In [10]:
# Updated paths based on your folder structure
HAZY_DIR = '/content/sots_data/indoor/hazy'
CLEAR_DIR = '/content/sots_data/indoor/clear'

# Initialize the dataset and loader
dataset = SOTSDataset(HAZY_DIR, CLEAR_DIR)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Success! Found {len(dataset)} image pairs in the indoor dataset. 🚀")

Success! Found 500 image pairs in the indoor dataset. 🚀


In [11]:
import numpy as np

def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    PIXEL_MAX = 1.0
    return 20 * torch.log10(PIXEL_MAX / torch.sqrt(mse))

# You can call this inside your training loop to show progress!

In [12]:
class SOTSDataset(Dataset):
    def __init__(self, hazy_path, clear_path, size=256):
        self.hazy_path = hazy_path
        self.clear_path = clear_path
        self.hazy_files = sorted(os.listdir(hazy_path))
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.hazy_files)

    def __getitem__(self, idx):
        hazy_filename = self.hazy_files[idx]
        # SOTS naming trick: "1_5.png" -> split by "_" -> take "1" -> add ".png"
        clear_filename = hazy_filename.split('_')[0] + '.png'

        hazy_img = Image.open(os.path.join(self.hazy_path, hazy_filename)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_filename)).convert('RGB')

        return self.transform(hazy_img), self.transform(clear_img)

# Re-initialize
dataset = SOTSDataset(HAZY_DIR, CLEAR_DIR)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"Dataset fixed! We now have {len(dataset)} training pairs. 🚀")

Dataset fixed! We now have 500 training pairs. 🚀


In [13]:
# Inside your train_oval_net loop, at the end of the epoch:
torch.save(model.state_dict(), 'oval_net_latest.pth')

In [14]:
from google.colab import files
files.download('oval_net_latest.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import torch.optim as optim

# 1. Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} 🚀")

# 2. Move your model to that device
model.to(device)

# 3. Define your loss function and optimizer
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Now you can run:
# train_oval_net(epochs=10)

Using device: cuda 🚀


In [16]:
import matplotlib.pyplot as plt

def visualize_results(model, loader, device):
    model.eval()
    hazy, clear = next(iter(loader))
    with torch.no_grad():
        output = model(hazy.to(device)).cpu()

    plt.figure(figsize=(15, 5))
    titles = ['Hazy Input', 'OVAL-Net Output', 'Ground Truth']
    imgs = [hazy[0], output[0], clear[0]]

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(titles[i])
        # Permute to (H, W, C) and clamp for display
        plt.imshow(imgs[i].permute(1, 2, 0).clamp(0, 1))
        plt.axis('off')
    plt.show()

# Call this after an epoch to see progress!
# visualize_results(model, train_loader, device)

In [17]:
def train_oval_net(epochs=10):
    model.train()
    print(f"Training started on {device}...")
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_psnr = 0.0

        for i, (hazy, clear) in enumerate(train_loader):
            hazy, clear = hazy.to(device), clear.to(device)

            optimizer.zero_grad()
            outputs = model(hazy)
            loss = criterion(outputs, clear)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_psnr += calculate_psnr(outputs, clear).item()

            # Real-time update every 20 batches
            if (i + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(train_loader)}], Current Loss: {loss.item():.4f}")

        avg_loss = epoch_loss / len(train_loader)
        avg_psnr = epoch_psnr / len(train_loader)
        print(f"==> Epoch [{epoch+1}/{epochs}] Final - Loss: {avg_loss:.4f} - PSNR: {avg_psnr:.2f}dB")

        # Save a checkpoint
        torch.save(model.state_dict(), f'ovalnet_epoch_{epoch+1}.pth')

# Make sure device is defined!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
train_oval_net(epochs=5)

Training started on cuda...
Epoch [1/5], Batch [20/125], Current Loss: 0.2190
Epoch [1/5], Batch [40/125], Current Loss: 0.1574
Epoch [1/5], Batch [60/125], Current Loss: 0.0980
Epoch [1/5], Batch [80/125], Current Loss: 0.1118
Epoch [1/5], Batch [100/125], Current Loss: 0.0949
Epoch [1/5], Batch [120/125], Current Loss: 0.0964
==> Epoch [1/5] Final - Loss: 0.1639 - PSNR: 12.63dB
Epoch [2/5], Batch [20/125], Current Loss: 0.0732
Epoch [2/5], Batch [40/125], Current Loss: 0.0915
Epoch [2/5], Batch [60/125], Current Loss: 0.0722
Epoch [2/5], Batch [80/125], Current Loss: 0.0925
Epoch [2/5], Batch [100/125], Current Loss: 0.0716
Epoch [2/5], Batch [120/125], Current Loss: 0.0752
==> Epoch [2/5] Final - Loss: 0.0876 - PSNR: 18.59dB
Epoch [3/5], Batch [20/125], Current Loss: 0.0719
Epoch [3/5], Batch [40/125], Current Loss: 0.0747
Epoch [3/5], Batch [60/125], Current Loss: 0.0683
Epoch [3/5], Batch [80/125], Current Loss: 0.0690
Epoch [3/5], Batch [100/125], Current Loss: 0.0564
Epoch [3/5]

In [18]:
import torch
import numpy as np

def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return 100
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def train_oval_net(epochs=10):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_psnr = 0.0

        for hazy, clear in train_loader:
            hazy, clear = hazy.to(device), clear.to(device)

            optimizer.zero_grad()
            outputs = model(hazy)
            loss = criterion(outputs, clear)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_psnr += calculate_psnr(outputs, clear).item()

        avg_loss = epoch_loss / len(train_loader)
        avg_psnr = epoch_psnr / len(train_loader)

        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f} - PSNR: {avg_psnr:.2f}dB")

# Start training
train_oval_net(epochs=10)

Epoch [1/10] - Loss: 0.0646 - PSNR: 21.13dB
Epoch [2/10] - Loss: 0.0603 - PSNR: 21.79dB
Epoch [3/10] - Loss: 0.0582 - PSNR: 22.00dB
Epoch [4/10] - Loss: 0.0561 - PSNR: 22.35dB
Epoch [5/10] - Loss: 0.0525 - PSNR: 22.91dB
Epoch [6/10] - Loss: 0.0519 - PSNR: 23.03dB
Epoch [7/10] - Loss: 0.0476 - PSNR: 23.69dB
Epoch [8/10] - Loss: 0.0497 - PSNR: 23.46dB
Epoch [9/10] - Loss: 0.0473 - PSNR: 23.85dB
Epoch [10/10] - Loss: 0.0441 - PSNR: 24.48dB


In [19]:
def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return 100
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

model.eval()
with torch.no_grad():
    h_test, c_test = next(iter(train_loader))
    out_test = model(h_test.to(device))
    score = calculate_psnr(out_test.cpu(), c_test)
    print(f"📊 Final Model PSNR Score: {score:.2f}dB")

📊 Final Model PSNR Score: 26.60dB


In [21]:
import torch
import torchvision.transforms as T
from PIL import Image

def final_dehaze(input_img):
    # 1. Force the model to evaluation mode (fixes Batchnorm/Dropout errors)
    model.eval()

    # 2. Ensure input is a PIL image
    if not isinstance(input_img, Image.Image):
        input_img = Image.fromarray(input_img)

    # 3. Robust Preprocessing
    test_transform = T.Compose([
        T.Resize((256, 256)), # Must match training size
        T.ToTensor()
    ])

    # 4. Move to GPU and add batch dimension
    img_t = test_transform(input_img).unsqueeze(0).to(device)

    # 5. Inference without tracking gradients (saves memory/prevents errors)
    with torch.no_grad():
        try:
            output = model(img_t)

            # 6. Post-process: Clamp values and move back to CPU
            output = output.squeeze(0).cpu().clamp(0, 1)
            return T.ToPILImage()(output)
        except Exception as e:
            return f"Runtime Error: {str(e)}"

# Update your Gradio Interface to use this new function
import gradio as gr
interface = gr.Interface(
    fn=final_dehaze,
    inputs=gr.Image(type="pil"),
    outputs="image"
)
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a135747b83527c532.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import gradio as gr

def final_dehaze(input_img):
    model.eval()

    # 1. Convert Gradio input (numpy) to PIL Image
    if isinstance(input_img, np.ndarray):
        input_img = Image.fromarray(input_img.astype('uint8'), 'RGB')

    # 2. Match the exact preprocessing used in training
    test_transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor()
    ])

    # 3. Prepare for GPU
    img_t = test_transform(input_img).unsqueeze(0).to(device)

    # 4. Run Model
    with torch.no_grad():
        output = model(img_t)
        # 5. Bring back to CPU and clean up
        output = output.squeeze(0).cpu().clamp(0, 1)
        return T.ToPILImage()(output)

# Launch with a shareable link for your teammates
interface = gr.Interface(
    fn=final_dehaze,
    inputs=gr.Image(),
    outputs="image",
    title="OVAL-Net: Outdoor Urban Dehazing"
)
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39f5e15907bf53727e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error